# N050 · K组反转、相交与随机指针

**目标：** 区分节点身份、引用拓扑与局部重连。

**先修：** N046, N048, N019。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** K组边界；指针换轨；深拷贝；随机边；映射与交织法。

## 从问题到算法

K组反转先确认还有完整k个节点，再反转这段并接回，否则不足一组的尾部必须保持原序。相交检测比较节点身份，两指针在走完自己链后切换到另一链，抵消长度差。随机指针复制则先建“原节点→新节点”映射，再复制next和random边，避免共享原节点。

## 最小实现

**本章接口：** `reverse_k_group(head, k)`、`get_intersection_node(a, b)`、`copy_random_list(head)`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values

def reverse_k_group(head,k):
    if k<1: raise ValueError('k must be positive')
    dummy=ListNode(0,head); before=dummy
    while True:
        end=before
        for _ in range(k):
            end=end.next
            if end is None: return dummy.next
        after=end.next; previous=after; current=before.next
        while current is not after:
            following=current.next; current.next=previous
            previous,current=current,following
        old_start=before.next; before.next=end; before=old_start

def get_intersection_node(a,b):
    p,q=a,b
    while p is not q:
        p=p.next if p else b
        q=q.next if q else a
    return p

class RandomNode:
    def __init__(self,val=0,next=None,random=None):
        self.val,self.next,self.random=val,next,random

def copy_random_list(head):
    mapping={None:None}; node=head
    while node:
        mapping[node]=RandomNode(node.val); node=node.next
    node=head
    while node:
        mapping[node].next=mapping[node.next]
        mapping[node].random=mapping[node.random]
        node=node.next
    return mapping[head]

## 正确性、前提与复杂度

整组确认后反转保持区间外边，组间前驱更新到旧组头。相交指针各走过A+B的长度，最终同步进入公共尾段或同到None。复制映射是一一对应，边的像保持拓扑。

**代价：** K组反转 O(n)时间/O(1)空间；相交 O(n+m)/O(1)；随机复制 O(n)时间空间。本章next链必须有限无环，random指向链内节点或None。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,3,4,5]
show_table(['k','重连后'],[(k,nodes_to_list(reverse_k_group(list_to_nodes(a),k))) for k in [1,2,3,6]])

k,重连后
1,"[1, 2, 3, 4, 5]"
2,"[2, 1, 4, 3, 5]"
3,"[3, 2, 1, 4, 5]"
6,"[1, 2, 3, 4, 5]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert nodes_to_list(reverse_k_group(list_to_nodes([1,2,3,4,5]),2))==[2,1,4,3,5]
assert nodes_to_list(reverse_k_group(list_to_nodes([1,2]),3))==[1,2]
shared=list_to_nodes([8,9]); a=ListNode(1,shared); b=ListNode(2,ListNode(3,shared))
assert get_intersection_node(a,b) is shared
assert get_intersection_node(list_to_nodes([1,2]),list_to_nodes([1,2])) is None
x,y,z=RandomNode(7),RandomNode(13),RandomNode(11); x.next=y; y.next=z
x.random=z; y.random=x; z.random=z
clone=copy_random_list(x); originals=[x,y,z]; clones=[clone,clone.next,clone.next.next]
assert not set(originals)&set(clones)
assert [n.val for n in clones]==[7,13,11]
assert clones[0].random is clones[2] and clones[1].random is clones[0] and clones[2].random is clones[2]
assert copy_random_list(None) is None
print('N050: 所有本章断言通过')

N050: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 不足K组保持原序。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较映射拷贝与交织拷贝的空间和副作用。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 25. Reverse Nodes in k-Group（canonical）
- 160. Intersection of Two Linked Lists（canonical）
- 138. Copy List with Random Pointer（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。